In [ ]:
"""Phi-3-mini SFT training script (Kaggle-safe, reproducible).

This is the best performing SFT training pipeline used in the project.
It trains a LoRA adapter on GSM8K, StrategyQA, and selected MMLU subjects, then evaluates
on random held-out subsets and saves checkpoints, CSVs, and plots.

Design goals:
- stable on Kaggle-style GPUs
- random sampling (not first-N slices)
- robust extraction for all three benchmarks
- LoRA / PEFT fine-tuning on Phi-3-mini-4k-instruct
- reproducible outputs
- minimal external dependencies

Notes:
- Uses open-weight backbone: microsoft/phi-3-mini-4k-instruct
- Uses structured prompts with <think> and <answer>
- Does NOT expose hidden chain-of-thought in outputs; training targets may include short rationale strings
"""

from __future__ import annotations

import os
import sys
import gc
import json
import math
import time
import random
import re
import types
import subprocess
import importlib.util
import importlib.machinery
from dataclasses import dataclass, asdict
from typing import Any, Dict, List, Optional, Tuple

# =============================================================================
# Environment bootstrap
# =============================================================================

def _stub_torchaudio() -> None:
    """Avoid torchaudio import issues in constrained notebook environments."""
    if "torchaudio" in sys.modules:
        return
    ta = types.ModuleType("torchaudio")
    ta.__dict__["__version__"] = "0.0.0"
    ta.__spec__ = importlib.machinery.ModuleSpec("torchaudio", loader=None)
    ta.__path__ = []
    sys.modules["torchaudio"] = ta
    for sm in ["_extension", "datasets", "functional", "models", "pipelines", "transforms", "utils"]:
        m = types.ModuleType(f"torchaudio.{sm}")
        m.__spec__ = importlib.machinery.ModuleSpec(f"torchaudio.{sm}", loader=None)
        m.__path__ = []
        setattr(ta, sm, m)
        sys.modules[f"torchaudio.{sm}"] = m


_stub_torchaudio()
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("TRANSFORMERS_NO_ADVISORY_WARNINGS", "true")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("PYDEVD_DISABLE_FILE_VALIDATION", "1")


def ensure_pkg(module_name: str, pip_name: str) -> None:
    if importlib.util.find_spec(module_name) is None:
        print(f"Installing missing dependency: {pip_name}", flush=True)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-qU", pip_name])


for mod, pip_name in [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("matplotlib", "matplotlib"),
    ("datasets", "datasets"),
    ("transformers", "transformers"),
    ("accelerate", "accelerate"),
    ("peft", "peft"),
    ("sentencepiece", "sentencepiece"),
    ("bitsandbytes", "bitsandbytes"),
]:
    try:
        ensure_pkg(mod, pip_name)
    except Exception as e:
        print(f"Dependency bootstrap warning for {pip_name}: {e}", flush=True)

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup,
)
from peft import LoraConfig, PeftModel, TaskType, get_peft_model, prepare_model_for_kbit_training

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update(
    {
        "figure.dpi": 140,
        "savefig.dpi": 180,
        "axes.titlesize": 13,
        "axes.labelsize": 11,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "legend.fontsize": 9,
        "font.family": "DejaVu Sans",
    }
)

# =============================================================================
# Config
# =============================================================================

@dataclass
class Config:
    base_model: str = "microsoft/phi-3-mini-4k-instruct"
    start_adapter: Optional[str] = None  # set to an existing adapter to continue from it
    out_dir: str = "/kaggle/working/phi3_first_sft"
    seed: int = 42

    # Hardware / memory
    use_4bit: bool = True
    fp16: bool = True
    gradient_checkpointing: bool = True

    # Data sizes
    gsm8k_train: int = 1200
    strategyqa_train: int = 900
    mmlu_train_per_subject: int = 28
    gsm8k_eval: int = 50
    strategyqa_eval: int = 50
    mmlu_eval_per_subject: int = 12

    # MMLU subjects used in the project
    mmlu_subjects: Tuple[str, ...] = (
        "abstract_algebra",
        "college_mathematics",
        "logical_fallacies",
        "formal_logic",
        "high_school_mathematics",
    )

    # Training
    num_epochs: int = 1
    train_batch_size: int = 1
    grad_accum: int = 8
    learning_rate: float = 5e-5
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    warmup_ratio: float = 0.05
    max_prompt_tokens: int = 1536

    # LoRA
    lora_r: int = 32
    lora_alpha: int = 64
    lora_dropout: float = 0.05

    # Generation / eval
    max_new_gsm8k: int = 224
    max_new_strategyqa: int = 80
    max_new_mmlu: int = 64
    repetition_penalty: float = 1.02
    self_consistency: int = 1  # set >1 if you want majority voting eval
    fewshot_gsm8k: int = 2
    fewshot_strategyqa: int = 2
    fewshot_mmlu: int = 3


CFG = Config()
OUT_DIR = CFG.out_dir
CSV_DIR = os.path.join(OUT_DIR, "csv")
PLOT_DIR = os.path.join(OUT_DIR, "plots")
CKPT_DIR = os.path.join(OUT_DIR, "checkpoints")
LOG_DIR = os.path.join(OUT_DIR, "logs")
CACHE_DIR = os.path.join(OUT_DIR, "cache")
for p in [OUT_DIR, CSV_DIR, PLOT_DIR, CKPT_DIR, LOG_DIR, CACHE_DIR]:
    os.makedirs(p, exist_ok=True)

SEED = CFG.seed
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE_COUNT = torch.cuda.device_count()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float16 if CFG.fp16 and torch.cuda.is_available() else torch.float32

def log(msg: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)

def cleanup(*objs):
    for o in objs:
        try:
            del o
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.synchronize()
        except Exception:
            pass

def save_df(df: pd.DataFrame, path: str):
    df.to_csv(path, index=False)
    log(f"Saved CSV -> {path}")

def save_json(obj: Any, path: str):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)
    log(f"Saved JSON -> {path}")

def save_plot(path: str):
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.close()
    log(f"Saved plot -> {path}")

def timer(label: str):
    class _T:
        def __enter__(self_inner):
            self_inner.t0 = time.time()
            log(f"{label} ...")
            return self_inner
        def __exit__(self_inner, exc_type, exc, tb):
            dt = time.time() - self_inner.t0
            if exc_type is None:
                log(f"{label} done in {dt:.1f}s")
            else:
                log(f"{label} failed after {dt:.1f}s: {exc_type.__name__}: {exc}")
    return _T()

# =============================================================================
# Sampling utilities
# =============================================================================

def sample_indices(n: int, k: int, seed: int, cache_name: str) -> List[int]:
    k = min(k, n)
    path = os.path.join(CACHE_DIR, f"{cache_name}_{k}_{seed}.json")
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    rng = np.random.RandomState(seed)
    idx = rng.choice(n, size=k, replace=False).tolist()
    with open(path, "w", encoding="utf-8") as f:
        json.dump(idx, f)
    return idx

def sample_dataset(ds, k: int, seed: int, cache_name: str):
    idx = sample_indices(len(ds), k, seed, cache_name)
    return ds.select(idx), idx

def preview_text(s: str, n: int = 120) -> str:
    s = str(s).replace("\n", " ")
    return s[:n] + ("..." if len(s) > n else "")

# =============================================================================
# Answer extraction / metrics
# =============================================================================

def canonical_number(pred: Optional[str]) -> Optional[str]:
    if pred is None:
        return None
    try:
        x = float(str(pred).replace(",", ""))
        return str(int(round(x))) if abs(x - round(x)) < 1e-8 else str(x)
    except Exception:
        return None

def is_number_correct(pred: Optional[str], gold: Optional[str]) -> bool:
    try:
        return abs(float(pred) - float(gold)) <= 1e-6
    except Exception:
        return False

def extract_number(text: str) -> Optional[str]:
    if not text:
        return None
    m = re.findall(r"<answer>(.*?)</answer>", text, flags=re.DOTALL | re.IGNORECASE)
    span = m[-1] if m else text
    boxed = re.findall(r"\\boxed\{([^}]*)\}", span)
    if boxed:
        nums = re.findall(r"-?\d+\.?\d*", boxed[-1].replace(",", ""))
        if nums:
            return nums[-1]
    nums = re.findall(r"-?\d+\.?\d*", span.replace(",", ""))
    return nums[-1] if nums else None

def extract_yes_no(text: str) -> Optional[str]:
    if not text:
        return None
    m = re.findall(r"<answer>(.*?)</answer>", text, flags=re.DOTALL | re.IGNORECASE)
    span = m[-1] if m else text
    hits = re.findall(r"\b(yes|no)\b", span, flags=re.IGNORECASE)
    if hits:
        return hits[-1].lower()
    hits = re.findall(r"\b(yes|no)\b", text, flags=re.IGNORECASE)
    return hits[-1].lower() if hits else None

def extract_mcq(text: str) -> Optional[str]:
    if not text:
        return None
    m = re.findall(r"<answer>(.*?)</answer>", text, flags=re.DOTALL | re.IGNORECASE)
    span = m[-1] if m else text
    up = span.upper().strip()
    if up in ["A", "B", "C", "D"]:
        return up
    for pat in [r"ANSWER\s*[:\-]?\s*\(?([ABCD])\)?", r"<ANSWER>\s*\(?([ABCD])\)?", r"\b([ABCD])\b\s*$"]:
        hits = re.findall(pat, up)
        if hits:
            return hits[-1].upper()
    hits = re.findall(r"\b([ABCD])\b", up)
    return hits[-1].upper() if hits else None

def extract_answer(text: str, task: str) -> Optional[str]:
    if not text:
        return None
    if task == "gsm8k":
        return extract_number(text)
    if task == "strategyqa":
        return extract_yes_no(text)
    if task == "mmlu":
        return extract_mcq(text)
    return text.strip()

def answer_present(text: str) -> bool:
    return bool(re.search(r"\b(yes|no|[ABCD])\b", str(text), flags=re.IGNORECASE))

# =============================================================================
# Task data loading
# =============================================================================

def get_mmlu_choices(sample: Dict[str, Any]) -> List[str]:
    choices = sample.get("choices")
    if choices is None:
        return []
    if isinstance(choices, str):
        try:
            parsed = json.loads(choices)
            return parsed if isinstance(parsed, list) else []
        except Exception:
            return []
    return list(choices)

def safe_mmlu_gold(sample: Dict[str, Any]) -> Optional[str]:
    g = sample.get("gold")
    if g is None:
        return None
    g = str(g).strip().upper()
    return g if g in ["A", "B", "C", "D"] else None

def load_gsm8k_split(split: str, n: int, cache_name: str):
    # Prefer the canonical dataset name, but fall back if the hub version differs.
    for repo in ["openai/gsm8k", "gsm8k"]:
        try:
            ds = load_dataset(repo, "main", split=split)
            ds, idx = sample_dataset(ds, n, SEED, f"{cache_name}_{repo.replace('/', '_')}")
            rows = []
            for i, s in enumerate(ds):
                raw = s["answer"]
                gold = canonical_number(str(raw).split("####")[-1].strip())
                rows.append(
                    {
                        "task": "gsm8k",
                        "sample_id": f"gsm8k_{split}_{idx[i]}",
                        "source_index": idx[i],
                        "question": s["question"],
                        "gold": gold,
                        "raw_gold": raw,
                        "rationale": str(raw).split("####")[0].strip(),
                        "source_repo": repo,
                    }
                )
            return pd.DataFrame(rows)
        except Exception:
            continue
    return pd.DataFrame()

def load_strategyqa_split(split: str, n: int, cache_name: str):
    try:
        ds = load_dataset("ChilleD/StrategyQA", split=split)
        ds, idx = sample_dataset(ds, n, SEED, cache_name)
        rows = []
        for i, s in enumerate(ds):
            rows.append(
                {
                    "task": "strategyqa",
                    "sample_id": f"strategyqa_{split}_{idx[i]}",
                    "source_index": idx[i],
                    "question": s["question"],
                    "gold": "yes" if bool(s["answer"]) else "no",
                    "raw_gold": s["answer"],
                    "rationale": "",
                }
            )
        return pd.DataFrame(rows)
    except Exception:
        return pd.DataFrame()

def load_mmlu_subject_split(subject: str, split_candidates: Tuple[str, ...], n: int, cache_name: str):
    for split in split_candidates:
        try:
            ds = load_dataset("cais/mmlu", subject, split=split)
            ds, idx = sample_dataset(ds, n, SEED, f"{cache_name}_{subject}_{split}")
            rows = []
            for i, s in enumerate(ds):
                choices = get_mmlu_choices(s)
                if not choices and "choices" in s:
                    choices = list(s["choices"])
                gold = safe_mmlu_gold({"gold": chr(65 + int(s["answer"])) if isinstance(s["answer"], (int, np.integer)) else str(s["answer"])} )
                if gold is None and isinstance(s["answer"], (int, np.integer)):
                    gold = chr(65 + int(s["answer"]))
                rows.append(
                    {
                        "task": "mmlu",
                        "subject": subject,
                        "sample_id": f"mmlu_{subject}_{split}_{idx[i]}",
                        "source_index": idx[i],
                        "question": s["question"],
                        "choices": choices[:4],
                        "gold": gold,
                        "raw_gold": s["answer"],
                        "mmlu_split": split,
                        "rationale": "",
                    }
                )
            return pd.DataFrame(rows)
        except Exception:
            continue
    return pd.DataFrame()

def build_task_pools():
    log("Building task pools ...")
    pools: Dict[str, pd.DataFrame] = {}
    pools["gsm_train"] = load_gsm8k_split("train", CFG.gsm8k_train, "gsm8k_train")
    pools["qa_train"] = load_strategyqa_split("train", CFG.strategyqa_train, "strategyqa_train")

    mmlu_parts = []
    for subject in CFG.mmlu_subjects:
        part = load_mmlu_subject_split(subject, ("dev", "validation", "train", "test"), CFG.mmlu_train_per_subject, "mmlu_train")
        if len(part) > 0:
            mmlu_parts.append(part)
    pools["mmlu_train"] = pd.concat(mmlu_parts, ignore_index=True) if mmlu_parts else pd.DataFrame()

    pools["gsm_eval"] = load_gsm8k_split("test", CFG.gsm8k_eval, "gsm8k_eval")
    pools["qa_eval"] = load_strategyqa_split("test", CFG.strategyqa_eval, "strategyqa_eval")

    mmlu_eval_parts = []
    for subject in CFG.mmlu_subjects:
        part = load_mmlu_subject_split(subject, ("test", "validation", "dev"), CFG.mmlu_eval_per_subject, "mmlu_eval")
        if len(part) > 0:
            mmlu_eval_parts.append(part)
    pools["mmlu_eval"] = pd.concat(mmlu_eval_parts, ignore_index=True) if mmlu_eval_parts else pd.DataFrame()
    return pools

# =============================================================================
# Prompting
# =============================================================================

def build_chat_prompt(tokenizer, system: str, user: str) -> str:
    if hasattr(tokenizer, "apply_chat_template"):
        try:
            msgs = [
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ]
            return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        except Exception:
            pass
    return f"<|system|>\n{system}\n<|user|>\n{user}\n<|assistant|>\n"

def choose_fewshots(sample: Dict[str, Any], bank: Dict[str, Any], k: int) -> List[Dict[str, Any]]:
    if k <= 0:
        return []
    task = sample["task"]
    if task == "gsm8k":
        pool = bank.get("gsm8k", [])
    elif task == "strategyqa":
        pool = bank.get("strategyqa", [])
    else:
        subj = sample.get("subject", "")
        pool = bank.get("mmlu", {}).get(subj, [])
        if not pool and bank.get("mmlu"):
            pool = sum(bank["mmlu"].values(), [])
    
    if not pool:
        return []
        
    pool = [x for x in pool if x.get("sample_id") != sample.get("sample_id")]
    if not pool:
        return []
        
    seed = CFG.seed + abs(hash(str(sample.get("sample_id", "x")))) % 10000
    rng = random.Random(seed)
    return rng.sample(pool, min(k, len(pool)))

def build_prompt(tokenizer, sample: Dict[str, Any], fewshots: List[Dict[str, Any]]) -> str:
    system = "You are a careful reasoning assistant. Think privately and output the final answer in the required format."
    fs = ""
    if fewshots:
        blocks = []
        for ex in fewshots:
            if ex["task"] == "gsm8k":
                blocks.append(
                    f"Question: {ex['question']}\n"
                    f"<think>{ex.get('rationale','')}</think>\n"
                    f"<answer>{ex['gold']}</answer>"
                )
            elif ex["task"] == "strategyqa":
                blocks.append(f"Question: {ex['question']}\n<answer>{ex['gold']}</answer>")
            elif ex["task"] == "mmlu":
                opts = "\n".join([f"{chr(65+i)}. {c}" for i, c in enumerate(ex['choices'][:4])])
                blocks.append(f"Question: {ex['question']}\n{opts}\n<answer>{ex['gold']}</answer>")
        fs = "\n\n".join(blocks) + "\n\n"

    if sample["task"] == "gsm8k":
        user = (
            f"{fs}Question: {sample['question']}\n"
            "Solve step by step. Put your final numeric answer in <answer></answer>."
        )
    elif sample["task"] == "strategyqa":
        user = (
            f"{fs}Question: {sample['question']}\n"
            "Answer yes or no. Put only yes or no inside <answer></answer>."
        )
    elif sample["task"] == "mmlu":
        choice_txt = "\n".join([f"{chr(65+i)}. {c}" for i, c in enumerate(sample['choices'][:4])])
        user = (
            f"{fs}Question: {sample['question']}\n{choice_txt}\n"
            "Choose one option A, B, C, or D. Put only that letter inside <answer></answer>."
        )
    else:
        user = (
            f"{fs}Question: {sample['question']}\n"
            "Provide a short reasoning summary, then the final answer in <answer></answer>."
        )

    return build_chat_prompt(tokenizer, system, user)

def render_target_completion(sample: Dict[str, Any]) -> str:
    if sample["task"] == "gsm8k":
        return f"<think>{sample.get('rationale', '')}</think>\n<answer>{sample['gold']}</answer>"
    if sample["task"] == "strategyqa":
        return f"<answer>{sample['gold']}</answer>"
    if sample["task"] == "mmlu":
        return f"<answer>{sample['gold']}</answer>"
    return f"<answer>{sample['gold']}</answer>"

# =============================================================================
# Model loading / LoRA
# =============================================================================

def load_tokenizer(model_name: str):
    tok = AutoTokenizer.from_pretrained(model_name, trust_remote_code=False)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return tok

def discover_lora_targets(model) -> List[str]:
    candidates = ["qkv_proj", "o_proj", "gate_up_proj", "up_proj", "down_proj"]
    names = set()
    for n, m in model.named_modules():
        if isinstance(m, torch.nn.Linear):
            leaf = n.split(".")[-1]
            if leaf in candidates:
                names.add(leaf)
    return sorted(names) if names else candidates

def normalize_rope_scaling(cfg):
    rs = getattr(cfg, "rope_scaling", None)
    if isinstance(rs, dict):
        if "type" not in rs and "rope_type" in rs:
            rs["type"] = rs["rope_type"]
        if "rope_type" not in rs and "type" in rs:
            rs["rope_type"] = rs["type"]
        cfg.rope_scaling = rs
    return cfg

def load_base_model(base_model: str, tokenizer):
    log(f"Loading base model: {base_model}")
    quant_cfg = None
    if CFG.use_4bit and torch.cuda.is_available() and importlib.util.find_spec("bitsandbytes") is not None:
        quant_cfg = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16 if CFG.fp16 else torch.bfloat16,
        )

    load_kwargs = dict(
        low_cpu_mem_usage=True,
        torch_dtype=DTYPE,
        attn_implementation="eager",
    )
    if quant_cfg is not None:
        load_kwargs["quantization_config"] = quant_cfg

    try:
        cfg = AutoConfig.from_pretrained(base_model, trust_remote_code=False)
        cfg = normalize_rope_scaling(cfg)
        model = AutoModelForCausalLM.from_pretrained(base_model, config=cfg, trust_remote_code=False, **load_kwargs)
    except Exception as e:
        log(f"Retrying with trust_remote_code=True due to: {type(e).__name__}: {e}")
        cfg = AutoConfig.from_pretrained(base_model, trust_remote_code=True)
        cfg = normalize_rope_scaling(cfg)
        model = AutoModelForCausalLM.from_pretrained(base_model, config=cfg, trust_remote_code=True, **load_kwargs)

    model.config.use_cache = False
    return model

def attach_lora(model):
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=CFG.gradient_checkpointing)
    targets = discover_lora_targets(model)
    log(f"LoRA targets: {targets}")
    lcfg = LoraConfig(
        r=CFG.lora_r,
        lora_alpha=CFG.lora_alpha,
        lora_dropout=CFG.lora_dropout,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
        target_modules=targets,
    )
    model = get_peft_model(model, lcfg)
    if CFG.gradient_checkpointing and hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()
    try:
        model.print_trainable_parameters()
    except Exception:
        pass
    return model

# =============================================================================
# Training / supervised loss
# =============================================================================

def tokenize_example(tokenizer, prompt: str, completion: str):
    prompt_ids = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=CFG.max_prompt_tokens)["input_ids"][0]
    comp_ids = tokenizer(completion, add_special_tokens=False, return_tensors="pt")["input_ids"][0]
    input_ids = torch.cat([prompt_ids, comp_ids], dim=0).unsqueeze(0).to(DEVICE)
    labels = torch.full_like(input_ids, -100)
    labels[0, prompt_ids.shape[0] :] = comp_ids.to(DEVICE)
    attention_mask = torch.ones_like(input_ids)
    return input_ids, attention_mask, labels, prompt_ids.shape[0], comp_ids

def supervised_loss(model, batch, tokenizer):
    out = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        labels=batch["labels"],
        output_hidden_states=True,
        return_dict=True,
    )
    ce = out.loss

    route_terms = []
    norm_mod = getattr(getattr(model, "model", None), "norm", None)
    if norm_mod is None:
        norm_mod = getattr(model, "norm", None)
    lm_head = getattr(model, "lm_head", None)
    if lm_head is None:
        lm_head = getattr(model, "embed_out", None)

    for i, meta in enumerate(batch["meta"]):
        hs = out.hidden_states
        if hs is None or lm_head is None:
            continue
        answer_pos = min(batch["answer_pos"][i], hs[-1].shape[1] - 1)
        target = meta["gold"]
        if meta["task"] == "gsm8k":
            gold_ids = []
            for t in [str(int(float(target))) if target not in [None, ""] else "0", target]:
                if t is None:
                    continue
                enc = tokenizer.encode(str(t), add_special_tokens=False)
                if len(enc) == 1:
                    gold_ids.append(enc[0])
            gold_ids = sorted(set(gold_ids))
        elif meta["task"] == "strategyqa":
            gold_ids = []
            for t in [target, f" {target}"]:
                enc = tokenizer.encode(str(t), add_special_tokens=False)
                if len(enc) == 1:
                    gold_ids.append(enc[0])
            gold_ids = sorted(set(gold_ids))
        else:
            gold_ids = []
            for t in [target, f" {target}"]:
                enc = tokenizer.encode(str(t), add_special_tokens=False)
                if len(enc) == 1:
                    gold_ids.append(enc[0])
            gold_ids = sorted(set(gold_ids))

        if not gold_ids:
            continue

        num_layers = len(hs)
        target_curve = torch.sigmoid(
            (torch.arange(num_layers, device=DEVICE, dtype=torch.float32) - 0.72 * (num_layers - 1)) / max(1.5, 0.12 * num_layers)
        )
        probs = []
        for h in hs:
            vec = h[i, answer_pos, :]
            if norm_mod is not None:
                vec = norm_mod(vec)
            logits = lm_head(vec)
            p = torch.softmax(logits.float(), dim=-1)
            probs.append(max(float(p[j].item()) for j in gold_ids if j < p.shape[-1]))
        probs = torch.tensor(probs, device=DEVICE)
        route_terms.append(F.mse_loss(probs, target_curve))

    route = torch.stack(route_terms).mean() if route_terms else torch.tensor(0.0, device=ce.device)
    total = ce + 0.2 * route
    return total, {"ce": float(ce.item()), "route": float(route.item())}

def build_fewshot_bank(train_df: pd.DataFrame) -> Dict[str, Any]:
    bank = {"gsm8k": [], "strategyqa": [], "mmlu": {}}
    if len(train_df) == 0:
        return bank
    gsm = train_df[train_df.task == "gsm8k"].sample(min(40, (train_df.task == "gsm8k").sum()), random_state=SEED)
    qa = train_df[train_df.task == "strategyqa"].sample(min(40, (train_df.task == "strategyqa").sum()), random_state=SEED)
    mmlu = train_df[train_df.task == "mmlu"].sample(min(40, (train_df.task == "mmlu").sum()), random_state=SEED)
    bank["gsm8k"] = gsm.to_dict("records")
    bank["strategyqa"] = qa.to_dict("records")
    for subj, g in mmlu.groupby("subject"):
        bank["mmlu"][subj] = g.to_dict("records")
    return bank

def attach_prompt_and_target(df: pd.DataFrame, tokenizer, fewshot_bank: Dict[str, Any], hard_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, r in df.iterrows():
        sample = r.to_dict()
        if sample["task"] == "mmlu" and not isinstance(sample.get("choices"), list):
            sample["choices"] = ["A", "B", "C", "D"]
        fs = choose_fewshots(sample, fewshot_bank, CFG.fewshot_gsm8k if sample["task"] == "gsm8k" else CFG.fewshot_strategyqa if sample["task"] == "strategyqa" else CFG.fewshot_mmlu)
        prompt = build_prompt(tokenizer, sample, fs)
        completion = render_target_completion(sample)
        hard_weight = 1.0
        if len(hard_df) > 0 and sample.get("sample_id") in set(hard_df.get("sample_id", [])):
            hard_weight = 1.5
        rows.append(
            {
                "task": sample["task"],
                "subject": sample.get("subject", sample["task"]),
                "sample_id": sample["sample_id"],
                "question": sample["question"],
                "gold": sample["gold"],
                "prompt": prompt,
                "completion": completion,
                "target": completion,
                "fewshot_count": len(fs),
                "hard_weight": hard_weight,
                "meta": sample,
            }
        )
    return pd.DataFrame(rows)

# =============================================================================
# Evaluation
# =============================================================================

@torch.no_grad()
def generate_one(model, tokenizer, prompt: str, task: str, max_new: int):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=CFG.max_prompt_tokens).to(DEVICE)
    gen_kwargs = {
        "max_new_tokens": max_new,
        "do_sample": False,
        "repetition_penalty": CFG.repetition_penalty,
        "pad_token_id": tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }
    out = model.generate(**inputs, **gen_kwargs)
    full = tokenizer.decode(out[0], skip_special_tokens=True)
    completion = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return full, completion

def score_completion(sample: Dict[str, Any], completion: str, pred: Optional[str]) -> Dict[str, float]:
    task = sample["task"]
    format_ok = answer_present(completion)
    if task == "gsm8k":
        gold = canonical_number(sample["gold"])
        p = canonical_number(pred)
        exact = 1.0 if (p is not None and gold is not None and is_number_correct(p, gold)) else 0.0
        return {"reward": exact + (0.15 if format_ok else -0.1), "exact": exact, "format": float(format_ok)}
    if task == "strategyqa":
        exact = 1.0 if pred == sample["gold"] else 0.0
        return {"reward": exact + (0.10 if format_ok else -0.1), "exact": exact, "format": float(format_ok)}
    if task == "mmlu":
        exact = 1.0 if pred == sample["gold"] else 0.0
        return {"reward": exact + (0.10 if format_ok else -0.1), "exact": exact, "format": float(format_ok)}
    return {"reward": 0.0, "exact": 0.0, "format": 0.0}

def eval_task_pool(model, tokenizer, df: pd.DataFrame, task: str, fewshot_bank: Dict[str, Any], stage: str):
    if len(df) == 0:
        return pd.DataFrame(), 0.0

    rows = []
    max_new = {
        "gsm8k": CFG.max_new_gsm8k,
        "strategyqa": CFG.max_new_strategyqa,
        "mmlu": CFG.max_new_mmlu,
    }[task]

    for _, r in df.iterrows():
        sample = r.to_dict()
        fs = choose_fewshots(sample, fewshot_bank, CFG.fewshot_gsm8k if task == "gsm8k" else CFG.fewshot_strategyqa if task == "strategyqa" else CFG.fewshot_mmlu)
        prompt = build_prompt(tokenizer, sample, fs)
        full, comp = generate_one(model, tokenizer, prompt, task, max_new)
        pred = extract_answer(comp, task)
        if pred is None:
            pred = extract_answer(full, task)
        sc = score_completion(sample, comp, pred)
        correct = sc["exact"] > 0.5
        rows.append(
            {
                **sample,
                "stage": stage,
                "prediction": pred,
                "correct": bool(correct),
                "completion": comp,
                "full_output": full,
                "reward": sc["reward"],
                "format_ok": sc["format"],
            }
        )

    out = pd.DataFrame(rows)
    acc = float(out["correct"].mean())
    return out, acc

def evaluate_all(model, tokenizer, eval_sets: Dict[str, pd.DataFrame], fewshot_bank: Dict[str, Any], stage: str):
    results = {}
    all_rows = []
    for task, key in [("gsm8k", "gsm_eval"), ("strategyqa", "qa_eval"), ("mmlu", "mmlu_eval")]:
        df = eval_sets.get(key, pd.DataFrame())
        pred_df, acc = eval_task_pool(model, tokenizer, df, task, fewshot_bank, stage)
        results[task] = acc
        if len(pred_df) > 0:
            all_rows.append(pred_df)
            save_df(pred_df, os.path.join(CSV_DIR, f"{stage}_{task}_predictions.csv"))
    results["macro"] = float(np.mean([results.get("gsm8k", 0.0), results.get("strategyqa", 0.0), results.get("mmlu", 0.0)]))
    return results, (pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame())

# =============================================================================
# Plotting
# =============================================================================

def plot_training_curves(hist_df: pd.DataFrame, out_path: str):
    if len(hist_df) == 0:
        return
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    items = [
        ("loss", "Loss", (0, 0)),
        ("ce", "Cross-entropy", (0, 1)),
        ("route", "Routing loss", (1, 0)),
        ("eval_macro", "Eval macro accuracy", (1, 1)),
    ]
    for col, title, (i, j) in items:
        ax = axes[i, j]
        if col in hist_df.columns:
            ax.plot(hist_df["step"], hist_df[col], marker="o", linewidth=1.5)
        ax.set_title(title)
        ax.set_xlabel("Step")
    save_plot(out_path)

def plot_accuracy_progress(progress_rows: pd.DataFrame, out_path: str):
    if len(progress_rows) == 0:
        return
    fig, ax = plt.subplots(figsize=(10.5, 5.2))
    for task in progress_rows["task"].unique():
        sub = progress_rows[progress_rows.task == task].sort_values("stage_idx")
        ax.plot(sub["stage_name"], sub["accuracy"], marker="o", linewidth=2, label=task)
        for _, r in sub.iterrows():
            ax.text(r["stage_name"], r["accuracy"] + 0.008, f"{r['accuracy']:.2f}", ha="center", fontsize=8)
    ax.set_ylim(0, 1.05)
    ax.set_title("Task accuracy progression over checkpoints")
    ax.set_xlabel("Model version")
    ax.set_ylabel("Accuracy")
    ax.legend()
    save_plot(out_path)

# =============================================================================
# Training loop
# =============================================================================

def train_sft_stage(model, tokenizer, train_df: pd.DataFrame, eval_sets: Dict[str, pd.DataFrame], hard_df: pd.DataFrame, fewshot_bank: Dict[str, Any]):
    log("===== SFT WARMUP STAGE =====")
    model.train()
    optim = torch.optim.AdamW(model.parameters(), lr=CFG.learning_rate, weight_decay=CFG.weight_decay)
    steps_per_epoch = math.ceil(len(train_df) / CFG.train_batch_size)
    total_steps = steps_per_epoch * CFG.num_epochs
    warmup_steps = max(1, int(total_steps * CFG.warmup_ratio))
    scheduler = get_linear_schedule_with_warmup(optim, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

    task_groups = {t: train_df[train_df.task == t].to_dict("records") for t in train_df.task.unique()}
    task_list = [t for t in ["gsm8k", "strategyqa", "mmlu"] if t in task_groups and len(task_groups[t]) > 0]
    random.shuffle(task_list)

    history = []
    step = 0
    optim.zero_grad(set_to_none=True)

    for epoch in range(CFG.num_epochs):
        random.shuffle(task_list)
        epoch_rows = []
        for t in task_list:
            rows = task_groups[t].copy()
            random.shuffle(rows)
            epoch_rows.extend(rows)
        random.shuffle(epoch_rows)

        for i in range(0, len(epoch_rows), CFG.train_batch_size):
            batch_rows = epoch_rows[i : i + CFG.train_batch_size]
            if not batch_rows:
                continue

            input_ids_list = []
            attn_list = []
            labels_list = []
            answer_pos_list = []
            meta_list = []

            for ex in batch_rows:
                sample = ex.copy()
                if sample["task"] == "mmlu" and not isinstance(sample.get("choices"), list):
                    sample["choices"] = ["A", "B", "C", "D"]
                fewshots = choose_fewshots(sample, fewshot_bank, CFG.fewshot_gsm8k if sample["task"] == "gsm8k" else CFG.fewshot_strategyqa if sample["task"] == "strategyqa" else CFG.fewshot_mmlu)
                prompt = build_prompt(tokenizer, sample, fewshots)
                completion = render_target_completion(sample)
                ids = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=CFG.max_prompt_tokens)["input_ids"][0]
                comp = tokenizer(completion, add_special_tokens=False, return_tensors="pt")["input_ids"][0]
                seq = torch.cat([ids, comp], dim=0)
                lab = torch.full_like(seq, -100)
                lab[ids.shape[0] :] = comp
                input_ids_list.append(seq)
                attn_list.append(torch.ones_like(seq))
                labels_list.append(lab)
                answer_pos_list.append(ids.shape[0] + max(0, comp.shape[0] - 1))
                meta_list.append(sample)

            max_len = max(x.shape[0] for x in input_ids_list)
            def pad(xs, pad_value):
                out = []
                for x in xs:
                    if x.shape[0] < max_len:
                        x = torch.cat([x, torch.full((max_len - x.shape[0],), pad_value, dtype=x.dtype)], dim=0)
                    out.append(x)
                return torch.stack(out, dim=0)

            batch = {
                "input_ids": pad(input_ids_list, tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id).to(DEVICE),
                "attention_mask": pad(attn_list, 0).to(DEVICE),
                "labels": pad(labels_list, -100).to(DEVICE),
                "answer_pos": answer_pos_list,
                "meta": meta_list,
            }

            loss, metrics = supervised_loss(model, batch, tokenizer)
            loss = loss / CFG.grad_accum
            loss.backward()

            if (step + 1) % CFG.grad_accum == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)
                optim.step()
                scheduler.step()
                optim.zero_grad(set_to_none=True)

            history.append(
                {
                    "step": step,
                    "epoch": epoch,
                    "loss": float(loss.item()),
                    **metrics,
                }
            )

            if (step + 1) % 50 == 0:
                model.eval()
                eval_metrics, _ = evaluate_all(model, tokenizer, eval_sets, fewshot_bank, stage=f"sft_step_{step+1}")
                model.train()
                history[-1]["eval_gsm8k"] = eval_metrics.get("gsm8k", 0.0)
                history[-1]["eval_strategyqa"] = eval_metrics.get("strategyqa", 0.0)
                history[-1]["eval_mmlu"] = eval_metrics.get("mmlu", 0.0)
                history[-1]["eval_macro"] = eval_metrics.get("macro", 0.0)
                log(
                    f"[SFT] step {step+1} | loss={float(loss.item() * CFG.grad_accum):.4f} | "
                    f"macro={eval_metrics['macro']:.3f} | gsm8k={eval_metrics['gsm8k']:.3f} | "
                    f"strategyqa={eval_metrics['strategyqa']:.3f} | mmlu={eval_metrics['mmlu']:.3f}"
                )

            step += 1

    hist_df = pd.DataFrame(history)
    save_df(hist_df, os.path.join(CSV_DIR, "sft_history.csv"))
    plot_training_curves(hist_df, os.path.join(PLOT_DIR, "sft_curves.png"))
    return model, hist_df


# =============================================================================
# Main
# =============================================================================

def main():
    log("=======================================================")
    log("PHI-3 MINI FIRST SFT TRAINING")
    log("=======================================================")
    if torch.cuda.is_available():
        log(f"CUDA devices detected: {torch.cuda.device_count()}")
        for i in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(i)
            log(f"GPU {i}: {props.name} | {props.total_memory / (1024**3):.2f} GB")
    else:
        log("CUDA unavailable; running on CPU.")

    with timer("Loading task pools"):
        pools = build_task_pools()

    train_df = pd.concat([pools["gsm_train"], pools["qa_train"], pools["mmlu_train"]], ignore_index=True)
    eval_sets = {
        "gsm_eval": pools["gsm_eval"],
        "qa_eval": pools["qa_eval"],
        "mmlu_eval": pools["mmlu_eval"],
    }

    # Light hard-example weighting hook; fine to leave empty for first SFT.
    hard_df = pd.DataFrame()
    fewshot_bank = build_fewshot_bank(train_df)

    save_df(train_df, os.path.join(CSV_DIR, "train_pool.csv"))
    save_df(eval_sets["gsm_eval"], os.path.join(CSV_DIR, "gsm_eval_pool.csv"))
    save_df(eval_sets["qa_eval"], os.path.join(CSV_DIR, "strategyqa_eval_pool.csv"))
    save_df(eval_sets["mmlu_eval"], os.path.join(CSV_DIR, "mmlu_eval_pool.csv"))

    with timer("Loading tokenizer + model"):
        tokenizer = load_tokenizer(CFG.base_model)
        policy = load_base_model(CFG.base_model, tokenizer)
        if CFG.start_adapter and os.path.isdir(CFG.start_adapter) and os.path.exists(os.path.join(CFG.start_adapter, "adapter_config.json")):
            policy = prepare_model_for_kbit_training(policy, use_gradient_checkpointing=CFG.gradient_checkpointing)
            policy = PeftModel.from_pretrained(policy, CFG.start_adapter, is_trainable=True)
            if CFG.gradient_checkpointing and hasattr(policy, "gradient_checkpointing_enable"):
                policy.gradient_checkpointing_enable()
            try:
                policy.print_trainable_parameters()
            except Exception:
                pass
        else:
            policy = attach_lora(policy)
        policy.train()

    # Save initial baseline before any SFT updates.
    with timer("Baseline evaluation"):
        policy.eval()
        baseline_metrics, baseline_preds = evaluate_all(policy, tokenizer, eval_sets, fewshot_bank, stage="baseline")
        policy.train()
        
    save_json(baseline_metrics, os.path.join(OUT_DIR, "baseline_metrics.json"))
    save_df(baseline_preds, os.path.join(CSV_DIR, "baseline_predictions.csv"))
    log(f"Baseline macro accuracy: {baseline_metrics['macro']:.3f}")

    # SFT training.
    with timer("Training SFT stage"):
        policy, hist_df = train_sft_stage(policy, tokenizer, train_df, eval_sets, hard_df, fewshot_bank)

    # Final evaluation.
    with timer("Final evaluation"):
        policy.eval()
        final_metrics, final_preds = evaluate_all(policy, tokenizer, eval_sets, fewshot_bank, stage="sft_final")
    save_json(final_metrics, os.path.join(OUT_DIR, "final_metrics.json"))
    save_df(final_preds, os.path.join(CSV_DIR, "final_predictions.csv"))

    # Comparison table.
    comp = pd.DataFrame([
        {"stage": "baseline", **baseline_metrics},
        {"stage": "sft_final", **final_metrics},
    ])
    save_df(comp, os.path.join(CSV_DIR, "comparison_table.csv"))

    # Save final checkpoint.
    final_dir = os.path.join(CKPT_DIR, "sft_final")
    os.makedirs(final_dir, exist_ok=True)
    if hasattr(policy, "save_pretrained"):
        policy.save_pretrained(final_dir)
    tokenizer.save_pretrained(final_dir)
    log(f"Saved final checkpoint -> {final_dir}")

    # Plot comparison.
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(comp["stage"], comp["gsm8k"], marker="o", label="GSM8K")
    ax.plot(comp["stage"], comp["strategyqa"], marker="o", label="StrategyQA")
    ax.plot(comp["stage"], comp["mmlu"], marker="o", label="MMLU")
    ax.plot(comp["stage"], comp["macro"], marker="o", linewidth=2.5, label="Macro")
    ax.set_ylim(0, 1.05)
    ax.set_title("Baseline vs SFT accuracy")
    ax.set_ylabel("Accuracy")
    ax.legend()
    save_plot(os.path.join(PLOT_DIR, "accuracy_comparison.png"))

    print("\n===== BASELINE =====")
    print(pd.DataFrame([baseline_metrics]).to_string(index=False))
    print("\n===== FINAL SFT =====")
    print(pd.DataFrame([final_metrics]).to_string(index=False))
    print("\n===== COMPARISON =====")
    print(comp.to_string(index=False))
    print(f"\nOutputs saved to: {OUT_DIR}")

    cleanup(policy)


if __name__ == "__main__":
    main()